# Seq2Seq RNN — Dự báo xâm nhập mặn
**Kiến trúc:** Encoder–Decoder với Scheduled Sampling  
**Input:** 3 trạm (BenLuc + CauNoi + TanAn) + meteo + temporal + lag features  
**Target:** Salinity_BenLuc  
**Lookbacks:** 3, 6, 12 bước (6h, 12h, 24h)  
**Horizons:** 6, 12, 24 bước (12h, 24h, 48h)  
**Train:** 2020–2022 | **Val:** 2023 | **Test:** 2025


In [ ]:
import sys, os, itertools, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

DATA_DIR  = Path("data/final_dataset")
OUT_DIR   = Path("outputs/seq2seq")
PLOT_DIR  = OUT_DIR / "plots"
MODEL_DIR = OUT_DIR / "models"
for d in [PLOT_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 1. Cấu hình

In [ ]:
INPUT_STATIONS  = ["BenLuc", "CauNoi", "TanAn"]
TARGET_STATION  = "BenLuc"
TARGET_COL      = f"Salinity_{TARGET_STATION}"

TRAIN_YEARS = [2020, 2021, 2022]
VAL_YEARS   = [2023]
TEST_YEARS  = [2025]

FEATURE_COLS  = ["wind_speed", "temp", "total_precipitation"]
TEMPORAL_COLS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos", "month_sin", "month_cos"]
LAG_STEPS     = [6, 12, 24]

LOOKBACKS = [3, 6, 12]   # bước × 2h → 6h, 12h, 24h
HORIZONS  = [6, 12, 24]  # bước × 2h → 12h, 24h, 48h

# ── Hyperparameters ────────────────────────────────────────────────────
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
EPOCHS      = 100
PATIENCE    = 15
LR          = 1e-3
BATCH_SIZE  = 64

# ── Scheduled Sampling ─────────────────────────────────────────────────
# teacher_forcing_ratio bắt đầu = 1.0 (dùng 100% ground truth)
# giảm dần về TF_RATIO_END theo từng epoch
TF_RATIO_START = 1.0
TF_RATIO_END   = 0.1   # epoch cuối: chỉ dùng 10% ground truth


## 2. Tiện ích dữ liệu

In [ ]:
def load_station(name):
    return pd.read_csv(DATA_DIR / f"{name}_clean.csv", parse_dates=["Time"])

def split_years(df, years):
    return df[df["Time"].dt.year.isin(years)].copy()

def add_features(df, sal_col):
    df = df.copy()
    hour = df["Time"].dt.hour
    doy  = df["Time"].dt.dayofyear
    mon  = df["Time"].dt.month
    df["hour_sin"]  = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * hour / 24)
    df["doy_sin"]   = np.sin(2 * np.pi * doy  / 365)
    df["doy_cos"]   = np.cos(2 * np.pi * doy  / 365)
    df["month_sin"] = np.sin(2 * np.pi * mon  / 12)
    df["month_cos"] = np.cos(2 * np.pi * mon  / 12)
    for lag in LAG_STEPS:
        df[f"lag_{lag}"] = df[sal_col].shift(lag)
    return df.dropna().reset_index(drop=True)

def make_windows(features, target, lookback, horizon, timestamps=None):
    Xs, ys, ts = [], [], []
    for i in range(len(features) - lookback - horizon + 1):
        Xs.append(features[i: i+lookback])
        ys.append(target[i+lookback: i+lookback+horizon])
        if timestamps is not None:
            ts.append(timestamps[i+lookback: i+lookback+horizon])
    if not Xs:
        return (np.empty((0, lookback, features.shape[1]), np.float32),
                np.empty((0, horizon), np.float32),
                np.empty((0, horizon), dtype="datetime64[ns]"))
    return (np.array(Xs, np.float32),
            np.array(ys, np.float32),
            np.array(ts) if timestamps is not None
            else np.empty((len(Xs), horizon), dtype="datetime64[ns]"))

class Seq2SeqDataset(Dataset):
    """
    X     : (lookback, n_features)  — encoder input
    y     : (horizon,)              — decoder target (salinity thực tế)
    y_in  : (horizon,)              — decoder input (shifted: [SOS, y[0],...,y[-2]])
    """
    def __init__(self, X, y):
        self.X    = torch.tensor(X, dtype=torch.float32)
        self.y    = torch.tensor(y, dtype=torch.float32)
        # Decoder input: [0 (SOS token), y[0], y[1], ..., y[-2]]
        sos       = torch.zeros(X.shape[0], 1)
        self.y_in = torch.cat([sos, self.y[:, :-1]], dim=1)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i], self.y_in[i]

def build_dataset(lookback, horizon):
    """Merge 3 trạm, add features, build windows cho tất cả split."""
    sal_cols     = [f"Salinity_{s}" for s in INPUT_STATIONS]
    lag_cols     = [f"lag_{k}" for k in LAG_STEPS]
    feature_cols = sal_cols + FEATURE_COLS + TEMPORAL_COLS + lag_cols

    all_data = {s: load_station(s) for s in INPUT_STATIONS}

    result = {}
    scX = scY = None

    for split, years in [("train", TRAIN_YEARS),
                          ("val",   VAL_YEARS),
                          ("test",  TEST_YEARS)]:
        split_dfs = {s: split_years(all_data[s], years) for s in INPUT_STATIONS}

        # Merge 3 trạm
        merged = split_dfs["BenLuc"][["Time", "Salinity_BenLuc"]].copy()
        for s in ["CauNoi", "TanAn"]:
            merged = pd.merge(merged,
                              split_dfs[s][["Time", f"Salinity_{s}"]],
                              on="Time", how="inner")
        meteo  = split_dfs["BenLuc"][["Time"] + FEATURE_COLS]
        merged = pd.merge(merged, meteo, on="Time", how="inner")
        merged = merged.sort_values("Time").reset_index(drop=True)
        merged = add_features(merged, TARGET_COL)

        X_raw = merged[feature_cols].values.astype(np.float32)
        y_raw = merged[[TARGET_COL]].values.astype(np.float32)

        if split == "train":
            scX = StandardScaler(); scX.fit(X_raw)
            scY = StandardScaler(); scY.fit(y_raw)

        X_sc = scX.transform(X_raw)
        y_sc = scY.transform(y_raw).flatten()
        ts   = merged["Time"].values

        Xa, ya, ta = [], [], []
        for yr in sorted(merged["Time"].dt.year.unique()):
            m = merged["Time"].dt.year == yr
            Xw, yw, tw = make_windows(X_sc[m], y_sc[m], lookback, horizon, ts[m])
            Xa.append(Xw); ya.append(yw); ta.append(tw)

        result[split] = (np.concatenate(Xa),
                         np.concatenate(ya),
                         np.concatenate(ta))

    return result, scX, scY, feature_cols

# Kiểm tra nhanh
_ds, _, _, _fc = build_dataset(lookback=6, horizon=6)
_input_size = _ds["train"][0].shape[2]
print(f"✓ Input size : {_input_size} features")
print(f"  Train windows: {len(_ds['train'][0])}")
print(f"  Val   windows: {len(_ds['val'][0])}")
print(f"  Test  windows: {len(_ds['test'][0])}")


## 3. Kiến trúc Seq2Seq Encoder–Decoder

In [ ]:
class Encoder(nn.Module):
    """
    Nhận chuỗi lookback features → tạo context vector (hidden state cuối).
    """
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.rnn = nn.RNN(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, x):
        # x: (batch, lookback, input_size)
        outputs, hidden = self.rnn(x)
        # hidden: (num_layers, batch, hidden_size) — context vector
        return hidden


class Decoder(nn.Module):
    """
    Nhận context từ Encoder, dự báo từng bước một.
    Input mỗi bước: giá trị salinity bước trước (hoặc ground truth khi teacher forcing).
    """
    def __init__(self, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        # Input decoder: 1 giá trị salinity tại mỗi bước
        self.rnn = nn.RNN(
            1, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, hidden):
        # x: (batch, 1, 1) — input 1 bước
        out, hidden = self.rnn(x, hidden)
        pred = self.fc(out)   # (batch, 1, 1)
        return pred.squeeze(-1), hidden   # (batch, 1), hidden


class Seq2SeqRNN(nn.Module):
    """
    Encoder–Decoder với Scheduled Sampling.
    tf_ratio: teacher forcing ratio (1.0 = dùng 100% ground truth,
                                     0.0 = dùng 100% prediction)
    """
    def __init__(self, input_size, hidden_size, num_layers, horizon, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.encoder = Encoder(input_size, hidden_size, num_layers, dropout)
        self.decoder = Decoder(hidden_size, num_layers, dropout)

    def forward(self, x, y_true=None, tf_ratio=0.0):
        # ── Encoder ───────────────────────────────────────────────
        hidden = self.encoder(x)          # (num_layers, batch, hidden)
        batch  = x.size(0)

        # ── Decoder — autoregressive ───────────────────────────────
        outputs   = []
        dec_input = torch.zeros(batch, 1, 1).to(x.device)  # SOS token

        for t in range(self.horizon):
            pred, hidden = self.decoder(dec_input, hidden)  # pred: (batch, 1)
            outputs.append(pred)

            # Scheduled Sampling: dùng ground truth hay prediction?
            if y_true is not None and torch.rand(1).item() < tf_ratio:
                # Teacher forcing: dùng giá trị thực t
                dec_input = y_true[:, t].unsqueeze(1).unsqueeze(2)
            else:
                # Autoregressive: dùng giá trị vừa dự báo
                dec_input = pred.unsqueeze(2).detach()

        return torch.cat(outputs, dim=1)   # (batch, horizon)


## 4. Hàm huấn luyện với Scheduled Sampling

In [ ]:
def get_tf_ratio(epoch, total_epochs):
    """Giảm tuyến tính teacher forcing ratio từ START về END."""
    progress = epoch / total_epochs
    return TF_RATIO_START - progress * (TF_RATIO_START - TF_RATIO_END)

def train_seq2seq(model, tr_loader, val_loader, tag):
    model = model.to(DEVICE)
    criterion = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

    best_val, pat_cnt, best_state = float("inf"), 0, None
    history = {"train": [], "val": [], "tf_ratio": []}

    print(f"  {'Epoch':>6} | {'Train':>10} | {'Val':>10} | {'TF Ratio':>9} | Status")
    print(f"  {'-'*60}")

    for epoch in range(1, EPOCHS + 1):
        tf_ratio = get_tf_ratio(epoch, EPOCHS)
        history["tf_ratio"].append(tf_ratio)

        # ── Train ──────────────────────────────────────────────────
        model.train()
        tr_loss = 0.0
        for Xb, yb, _ in tr_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(Xb, y_true=yb, tf_ratio=tf_ratio)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_loss += loss.item() * len(Xb)
        tr_loss /= len(tr_loader.dataset)

        # ── Validation (không dùng teacher forcing) ────────────────
        model.eval()
        vl_loss = 0.0
        with torch.no_grad():
            for Xb, yb, _ in val_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                pred    = model(Xb, tf_ratio=0.0)   # autoregressive hoàn toàn
                vl_loss += criterion(pred, yb).item() * len(Xb)
        vl_loss /= len(val_loader.dataset)

        history["train"].append(tr_loss)
        history["val"].append(vl_loss)
        sch.step(vl_loss)

        if vl_loss < best_val:
            best_val   = vl_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat_cnt    = 0
            status     = f"✓ best={best_val:.6f}"
        else:
            pat_cnt += 1
            status   = f"no improve ({pat_cnt}/{PATIENCE})"

        if epoch % 5 == 0 or pat_cnt == PATIENCE:
            print(f"  {epoch:>6} | {tr_loss:>10.6f} | {vl_loss:>10.6f} | "
                  f"{tf_ratio:>9.3f} | {status}")

        if pat_cnt >= PATIENCE:
            print(f"  ⏹ Early stopping tại epoch {epoch}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, MODEL_DIR / f"{tag}.pt")
    return model, history


## 5. Đánh giá và trực quan hoá

In [ ]:
def evaluate_seq2seq(model, loader, scaler_y):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb, _ in loader:
            pred = model(Xb.to(DEVICE), tf_ratio=0.0)   # autoregressive
            preds.append(pred.cpu().numpy())
            trues.append(yb.numpy())
    preds = np.concatenate(preds)   # (N, horizon)
    trues = np.concatenate(trues)

    pi = scaler_y.inverse_transform(preds.reshape(-1,1)).reshape(preds.shape)
    ti = scaler_y.inverse_transform(trues.reshape(-1,1)).reshape(trues.shape)

    # Non-overlapping R²
    h      = pi.shape[1]
    idx    = np.arange(0, len(pi), h)
    flat_p = pi[idx].flatten()
    flat_t = ti[idx].flatten()

    # R² từng bước
    r2_steps = [float(r2_score(ti[:, s], pi[:, s])) for s in range(h)]

    return {
        "rmse"      : float(np.sqrt(mean_squared_error(flat_t, flat_p))),
        "mae"       : float(mean_absolute_error(flat_t, flat_p)),
        "r2"        : float(r2_score(flat_t, flat_p)),
        "r2_steps"  : r2_steps,
        "preds"     : pi,
        "trues"     : ti,
    }

def plot_results(history, test_m, tag, horizon, timestamps=None):
    fig = plt.figure(figsize=(16, 10), facecolor="#F5F5F5")
    fig.suptitle(tag, fontsize=11, color="#424242", fontweight="bold")

    gs = plt.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
    ax_loss = fig.add_subplot(gs[0, 0])
    ax_tf   = fig.add_subplot(gs[0, 1])
    ax_r2   = fig.add_subplot(gs[0, 2])
    ax_full = fig.add_subplot(gs[1, :2])
    ax_zoom = fig.add_subplot(gs[1, 2])

    # ── Loss curve ─────────────────────────────────────────────────
    ax_loss.plot(history["train"], label="Train", color="#1565C0", lw=1.3)
    ax_loss.plot(history["val"],   label="Val",   color="#E53935", lw=1.3)
    ax_loss.set_title("Loss curve", fontsize=10, fontweight="bold")
    ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("MSE")
    ax_loss.legend(fontsize=8); ax_loss.spines[["top","right"]].set_visible(False)
    ax_loss.yaxis.grid(True, color="#EEEEEE", lw=0.8); ax_loss.set_axisbelow(True)

    # ── Teacher forcing schedule ────────────────────────────────────
    ax_tf.plot(history["tf_ratio"], color="#2E7D32", lw=1.5)
    ax_tf.fill_between(range(len(history["tf_ratio"])),
                       history["tf_ratio"], alpha=0.2, color="#2E7D32")
    ax_tf.set_title("Scheduled Sampling", fontsize=10, fontweight="bold")
    ax_tf.set_xlabel("Epoch"); ax_tf.set_ylabel("Teacher Forcing Ratio")
    ax_tf.spines[["top","right"]].set_visible(False)
    ax_tf.yaxis.grid(True, color="#EEEEEE", lw=0.8); ax_tf.set_axisbelow(True)

    # ── R² từng bước ───────────────────────────────────────────────
    x_steps = [(s+1)*2 for s in range(horizon)]
    bar_clrs = ["#43A047" if r >= 0.7 else "#FFA726" if r >= 0.4
                else "#E53935" for r in test_m["r2_steps"]]
    bars = ax_r2.bar(x_steps, test_m["r2_steps"], color=bar_clrs,
                     width=1.5, edgecolor="white")
    for bar, r in zip(bars, test_m["r2_steps"]):
        ax_r2.text(bar.get_x() + bar.get_width()/2,
                   max(bar.get_height() + 0.02, 0.05),
                   f"{r:.3f}", ha="center", va="bottom", fontsize=8)
    ax_r2.axhline(0.7, color="#43A047", lw=1, ls="--", alpha=0.7)
    ax_r2.set_title("R² theo từng bước dự báo", fontsize=10, fontweight="bold")
    ax_r2.set_xlabel("Bước (giờ)"); ax_r2.set_ylabel("R²")
    ax_r2.set_xticks(x_steps)
    ax_r2.spines[["top","right"]].set_visible(False)
    ax_r2.yaxis.grid(True, color="#EEEEEE", lw=0.8); ax_r2.set_axisbelow(True)

    # ── Non-overlapping series (full test) ─────────────────────────
    h      = test_m["preds"].shape[1]
    idx    = np.arange(0, len(test_m["preds"]), h)
    p_no   = test_m["preds"][idx].flatten()
    t_no   = test_m["trues"][idx].flatten()
    n      = len(p_no)
    steps_per_day = 12
    n_days = n // steps_per_day
    if n_days > 0:
        p_day = p_no[:n_days*steps_per_day].reshape(n_days, steps_per_day).mean(1)
        t_day = t_no[:n_days*steps_per_day].reshape(n_days, steps_per_day).mean(1)
        ax_full.plot(t_day, label="Thực tế", color="#1565C0", lw=1.4)
        ax_full.plot(p_day, label="Dự báo",  color="#E53935", lw=1.2, alpha=0.85)
    ax_full.set_title(f"Toàn tập Test — trung bình ngày | R²={test_m['r2']:.3f}",
                      fontsize=10, fontweight="bold")
    ax_full.set_xlabel("Ngày"); ax_full.set_ylabel("Độ mặn TB (‰)")
    ax_full.legend(fontsize=8); ax_full.spines[["top","right"]].set_visible(False)
    ax_full.yaxis.grid(True, color="#EEEEEE", lw=0.8); ax_full.set_axisbelow(True)

    # ── Zoom 7 ngày đầu ────────────────────────────────────────────
    n_detail = min(84, len(test_m["preds"]))
    ax_zoom.plot(test_m["trues"][:n_detail, 0],
                 label="Thực tế", color="#1565C0", lw=1.3)
    ax_zoom.plot(test_m["preds"][:n_detail, 0],
                 label="Dự báo t+2h", color="#E53935", lw=1.1, alpha=0.85)
    ax_zoom.set_title("Chi tiết 7 ngày đầu (bước t+2h)", fontsize=10, fontweight="bold")
    ax_zoom.set_xlabel("Window"); ax_zoom.set_ylabel("Độ mặn (‰)")
    ax_zoom.legend(fontsize=8); ax_zoom.spines[["top","right"]].set_visible(False)
    ax_zoom.yaxis.grid(True, color="#EEEEEE", lw=0.8); ax_zoom.set_axisbelow(True)

    for ax in fig.get_axes():
        ax.set_facecolor("#FAFAFA")

    fig.savefig(PLOT_DIR / f"{tag}.png", dpi=120, bbox_inches="tight",
                facecolor="#F5F5F5")
    plt.close(fig)
    print(f"  ✓ Plot lưu: {tag}.png")


## 6. Chạy tất cả thí nghiệm

In [ ]:
results = []
combos  = list(itertools.product(LOOKBACKS, HORIZONS))
total   = len(combos)

print(f"Tổng số thí nghiệm: {total}  (Seq2Seq RNN — 3 trạm → BenLuc)")
print(f"{'='*65}")

for exp_idx, (lb, hz) in enumerate(combos, 1):
    lb_h = lb * 2; hz_h = hz * 2
    tag  = f"Seq2Seq_RNN_lb{lb_h}h_hz{hz_h}h"

    print(f"\n{'#'*65}")
    print(f"  Thí nghiệm {exp_idx}/{total}: lookback={lb_h}h → horizon={hz_h}h")
    print(f"{'#'*65}")

    # ── Build dataset ──────────────────────────────────────────────
    ds, scX, scY, feat_cols = build_dataset(lb, hz)
    input_size = ds["train"][0].shape[2]

    if any(len(ds[k][0]) == 0 for k in ["train","val","test"]):
        print("  SKIP — không đủ windows"); continue

    tr_loader  = DataLoader(
        Seq2SeqDataset(ds["train"][0], ds["train"][1]),
        BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(
        Seq2SeqDataset(ds["val"][0], ds["val"][1]),
        BATCH_SIZE)
    te_loader  = DataLoader(
        Seq2SeqDataset(ds["test"][0], ds["test"][1]),
        BATCH_SIZE)

    # ── Build model ────────────────────────────────────────────────
    model = Seq2SeqRNN(
        input_size  = input_size,
        hidden_size = HIDDEN_SIZE,
        num_layers  = NUM_LAYERS,
        horizon     = hz,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model params: {n_params:,}")

    # ── Train ──────────────────────────────────────────────────────
    t0 = time.time()
    model, history = train_seq2seq(model, tr_loader, val_loader, tag)
    elapsed = time.time() - t0

    # ── Evaluate ───────────────────────────────────────────────────
    val_m  = evaluate_seq2seq(model, val_loader, scY)
    test_m = evaluate_seq2seq(model, te_loader,  scY)

    # ── Plot ───────────────────────────────────────────────────────
    plot_results(history, test_m, tag, hz)

    # ── Print kết quả theo từng bước ─────────────────────────────
    print(f"\n  {'─'*55}")
    print(f"  KẾT QUẢ — lookback={lb_h}h, horizon={hz_h}h")
    print(f"  {'─'*55}")
    print(f"  {'Tập':>10} | {'RMSE':>8} | {'MAE':>8} | {'R²':>8}")
    print(f"  {'─'*55}")
    print(f"  {'Val':>10} | {val_m['rmse']:>8.4f} | {val_m['mae']:>8.4f} | {val_m['r2']:>8.4f}")
    print(f"  {'Test':>10} | {test_m['rmse']:>8.4f} | {test_m['mae']:>8.4f} | {test_m['r2']:>8.4f}")
    print(f"  {'─'*55}")
    print(f"  R² từng bước test:")
    for s, r2s in enumerate(test_m["r2_steps"]):
        bar = "█" * int(r2s * 20) if r2s > 0 else ""
        print(f"    t+{(s+1)*2:>2}h : {r2s:>6.3f}  {bar}")
    print(f"  Thời gian: {elapsed:.1f}s")

    results.append({
        "lookback_h" : lb_h, "horizon_h"  : hz_h,
        "val_RMSE"   : round(val_m["rmse"],  4),
        "val_MAE"    : round(val_m["mae"],   4),
        "val_R2"     : round(val_m["r2"],    4),
        "test_RMSE"  : round(test_m["rmse"], 4),
        "test_MAE"   : round(test_m["mae"],  4),
        "test_R2"    : round(test_m["r2"],   4),
        **{f"test_R2_step_{(s+1)*2}h": round(r, 4)
           for s, r in enumerate(test_m["r2_steps"])},
    })

# ── Lưu kết quả ────────────────────────────────────────────────────
df_res = pd.DataFrame(results)
df_res.to_csv(OUT_DIR  / "seq2seq_results.csv",   index=False)
df_res.to_excel(OUT_DIR / "seq2seq_results.xlsx",  index=False)
print(f"\n{'='*65}")
print(f"✅ Hoàn tất {len(results)} thí nghiệm")
print(f"   Kết quả: outputs/seq2seq/seq2seq_results.xlsx")


## 7. Tổng hợp kết quả

In [ ]:
df = pd.read_csv(OUT_DIR / "seq2seq_results.csv")

# ── Bảng pivot R² theo lookback × horizon ──────────────────────────
pivot = df.pivot(index="lookback_h", columns="horizon_h", values="test_R2")
print("Test R² — Seq2Seq RNN (3 trạm → BenLuc)")
print("="*45)
print(pivot.round(4).to_string())

# ── Heatmap ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5), facecolor="#F5F5F5")
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"Horizon\n{c}h" for c in pivot.columns], fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"Lookback {r}h" for r in pivot.index], fontsize=9)
ax.set_title("Test R² — Seq2Seq RNN\n(3 trạm → BenLuc)",
             fontsize=11, fontweight="bold")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                    fontsize=10, color="white" if v > 0.6 else "#212121",
                    fontweight="bold")
plt.colorbar(im, ax=ax, label="R²")
fig.tight_layout()
fig.savefig(PLOT_DIR / "summary_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n✓ Lưu: outputs/seq2seq/plots/summary_heatmap.png")

# Best config
best = df.loc[df["test_R2"].idxmax()]
print(f"\n★ Best config: lookback={best['lookback_h']}h, "
      f"horizon={best['horizon_h']}h → Test R²={best['test_R2']:.4f}")
